# AI Hardware Architecture

**Author:** Nitin Sarangdhar  
**Affiliation:** SemiX Visiting Professor of practice | IIT Bombay  
**Date:** March 2026  

Experiment: Architecture Stress Test (RNN vs. LSTM vs. Transformer)This notebook implements a "Memory Clue" challenge.

The task is simple: a model is given a sequence of numbers, and it must remember the very first number (the clue) to predict the output at the very end of the sequence.1.

The Challenge: "The Wall" of Sequence LengthWe test the models at two different sequence lengths:Length 15 (The Safe Zone): Short enough for most recurrent models to pass.Length 45 (The Wall): Long enough to trigger the Vanishing Gradient problem in basic RNNs.

The Contenders Architecture Description

Simple RNN: The baseline recurrent model.Suffers from the "vanishing gradient"—as the sequence grows, it "forgets" the beginning.

Simple LSTM: The gated solution (1997).Hardware Hack: We initialize the Forget Gate bias to 1.0. This ensures the "memory highway" is open by default at the start of training.

Transformer: The attention-based solution (2017).Positional Encoding: Since Transformers process everything in parallel, we add a spatial "map" so the model knows the order of the data.

Implementation Details

The Data Generator
The function get_batch creates random noise sequences. The "clue" is always at index 0. To win, the model must ignore all the noise from indices 1 to N and propagate the gradient all the way back to the start.

The Test Engine
Optimizer: We use Adam (Learning Rate: 0.005).

Loss Function: MSELoss (Mean Squared Error), suitable for this regression task.

Success Metric: A model is marked as "WIN" if the absolute difference between the expected clue and the prediction is less than 0.25.

Key Observations to Look For

Training Speed: Notice how the Transformer handles longer sequences compared to the RNN/LSTM.

Accuracy Decay: Watch the RNN status change from ✅ to ❌ as we move from length 15 to 45.

LSTM Resilience: Observe how the gated architecture (specifically the Cell State highway) allows the LSTM to pass the test even when the sequence is tripled in length.

In [1]:
import torch
import torch.nn as nn
import time
import math

# 1. THE DATA GENERATOR
def get_batch(seq_len, batch_size=64):
    x = torch.randn(batch_size, seq_len, 1)
    y = x[:, 0, :] # The "Clue" is at index 0
    return x, y

# 2. THE MODELS
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32, batch_first=True)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        # Initialize Forget Gate bias to 1.0 (The Hardware Hack)
        for name, param in self.lstm.named_parameters():
            if 'bias' in name:
                n = param.size(0)
                param.data[n//4:n//2].fill_(1.0)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # Transformers need a bit more 'heft' to handle 1D data
        self.input_proj = nn.Linear(1, 32)
        self.pos_encoder = nn.Parameter(torch.randn(1, 100, 32)) # Up to 100 steps
        encoder_layer = nn.TransformerEncoderLayer(d_model=32, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        x = self.input_proj(x) + self.pos_encoder[:, :x.size(1), :]
        out = self.transformer(x)
        return self.fc(out[:, -1, :])

# 3. TEST ENGINE
def run_comparison(model_class, seq_len):
    model = model_class()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()

    start_time = time.time()
    model.train()
    for i in range(1000): # Standardized training steps
        x, y = get_batch(seq_len)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
    duration = time.time() - start_time

    # Final Inference Check
    model.eval()
    with torch.no_grad():
        tx, ty = get_batch(seq_len, batch_size=1)
        pred_val = model(tx).item()
        target_val = ty.item()

    return duration, loss.item(), target_val, pred_val

# --- EXECUTION ---
seq_lengths = [15, 45] # Short (Safe) and Long (The Wall)
configs = [('RNN', SimpleRNN), ('LSTM', SimpleLSTM), ('TRNSF', SimpleTransformer)]

print(f"{'Model':<8} | {'Len':<4} | {'Time':<7} | {'Loss':<8} | {'Expected':<8} | {'Pred':<8} | {'Status'}")
print("-" * 80)

for length in seq_lengths:
    for name, m_class in configs:
        t, loss, exp, pred = run_comparison(m_class, length)
        status = "✅ WIN" if abs(exp - pred) < 0.25 else "❌ FAIL"
        print(f"{name:<8} | {length:<4} | {t:>5.2f}s | {loss:.4f} | {exp:>8.3f} | {pred:>8.3f} | {status}")

Model    | Len  | Time    | Loss     | Expected | Pred     | Status
--------------------------------------------------------------------------------
RNN      | 15   |  3.05s | 0.0123 |   -1.358 |   -1.293 | ✅ WIN
LSTM     | 15   |  3.94s | 0.0002 |   -1.204 |   -1.172 | ✅ WIN
TRNSF    | 15   | 57.03s | 0.0084 |    0.291 |    0.282 | ✅ WIN
RNN      | 45   |  6.57s | 1.2022 |    0.556 |   -0.012 | ❌ FAIL
LSTM     | 45   |  4.05s | 0.0049 |    1.383 |    1.494 | ✅ WIN
TRNSF    | 45   | 172.97s | 0.0077 |    0.248 |    0.294 | ✅ WIN


Note that the RNN, LSTM runtime is Order(Len) while Transformer runtime is Order(Len**2). The execution is performed on a CPU. Change the runtime to a GPU and you will see significant improvement in Transformer runtime due to paralellization of the execution

Below is an example of a real life use case where we can perform sentiment detection of a given sentence.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

# 1. VOCABULARY AND DECODER
vocab = {
    "<PAD>": 0, "i": 1, "was": 2, "felt": 3, "initially": 4,
    "happy": 5, "sad": 6, "but": 7, "then": 8, "the": 9,
    "service": 10, "was_": 11, "bad": 12, "good": 13,
    "neutral": 15, "same": 16
}
id_to_word = {v: k for k, v in vocab.items()}

def decode_sentence(token_ids):
    words = []
    for tid in token_ids.tolist():
        tid = int(tid)
        if tid in id_to_word:
            words.append(id_to_word[tid])
        elif tid >= 20:
            if not words or words[-1] != "[...]":
                words.append("[...]")
    return " ".join(words)

# 2. ARTIFICIALLY STRESSFUL GENERATOR (Fixed Variety)
def get_hardcoded_batch(batch_size=33, noise_len=300):
    data, targets = [], []
    # Ensure equal distribution of scenarios 0, 1, and 2
    for i in range(batch_size):
        scenario = i % 3

        if scenario == 0:
            start_part, end_part, target = [4, 1, 2, 5], [7, 8, 12], 0  # Happy -> Bad
        elif scenario == 1:
            start_part, end_part, target = [4, 1, 2, 6], [7, 8, 13], 1  # Sad -> Good
        else:
            start_part, end_part, target = [4, 1, 2, 15], [7, 8, 16], 2 # Neutral -> Same

        # Static long-distance noise to force vanishing gradient
        noise = [20] * noise_len
        full_seq = start_part + noise + end_part
        data.append(torch.tensor(full_seq))
        targets.append(target)

    return torch.stack(data), torch.tensor(targets)

# 3. MODELS
class EmotionStressModel(nn.Module):
    def __init__(self, arch_type, vocab_size=101, embed_dim=16, hidden_dim=16):
        super().__init__()
        self.arch_type = arch_type
        self.embed = nn.Embedding(vocab_size, embed_dim)

        if arch_type == 'RNN':
            self.core = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif arch_type == 'LSTM':
            self.core = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif arch_type == 'Transformer':
            layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=2,
                                               dim_feedforward=32, batch_first=True)
            self.core = nn.TransformerEncoder(layer, num_layers=1)

        self.classifier = nn.Linear(hidden_dim if arch_type != 'Transformer' else embed_dim, 3)

    def forward(self, x):
        x = self.embed(x)
        if self.arch_type in ['RNN', 'LSTM']:
            out, _ = self.core(x)
            return self.classifier(out[:, -1, :])
        else:
            out = self.core(x)
            return self.classifier(out.mean(dim=1))

# 4. EXECUTION
NOISE_LIMIT = 300
print(f"--- STEP 1: VISUALIZING THE ARTIFICIAL TEST CASES ({NOISE_LIMIT} tokens) ---")
# Generate one batch and show one of each type
sample_in, sample_lab = get_hardcoded_batch(batch_size=3, noise_len=NOISE_LIMIT)
label_map = {0: "Negative", 1: "Positive", 2: "Neutral"}
for i in range(3):
    print(f"Scenario {i+1}: {decode_sentence(sample_in[i])}")
    print(f"Target: {sample_lab[i].item()} ({label_map[sample_lab[i].item()]})\n")

print("--- STEP 2: ARCHITECTURE SHOWDOWN ---")
print(f"{'Architecture':<15} | {'Loss':<10} | {'Test Acc'}")
print("-" * 55)

for arch in ['RNN', 'LSTM', 'Transformer']:
    model = EmotionStressModel(arch)
    # Using SGD to prove that architecture (not optimizer tricks) matters
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for i in range(1000):
        inputs, labels = get_hardcoded_batch(noise_len=NOISE_LIMIT)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    test_in, test_lab = get_hardcoded_batch(batch_size=99, noise_len=NOISE_LIMIT)
    with torch.no_grad():
        acc = (model(test_in).argmax(dim=1) == test_lab).float().mean().item()

    status = "✅ WIN" if acc > 0.70 else "❌ FAIL"
    print(f"{arch:<15} | {loss:.6f} | {acc*100:>6.1f}% {status}")

--- STEP 1: VISUALIZING THE ARTIFICIAL TEST CASES (300 tokens) ---
Scenario 1: initially i was happy [...] but then bad
Target: 0 (Negative)

Scenario 2: initially i was sad [...] but then good
Target: 1 (Positive)

Scenario 3: initially i was neutral [...] but then same
Target: 2 (Neutral)

--- STEP 2: ARCHITECTURE SHOWDOWN ---
Architecture    | Loss       | Test Acc
-------------------------------------------------------
RNN             | 0.001233 |  100.0% ✅ WIN
LSTM            | 0.003299 |  100.0% ✅ WIN
Transformer     | 0.000654 |  100.0% ✅ WIN


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

# 1. VOCABULARY
vocab = {
    "<PAD>": 0, "i": 1, "was": 2, "initially": 3, "happy": 4,
    "sad": 5, "neutral": 6, "but": 7, "then": 8, "it": 9,
    "changed": 10, "stayed": 11, "the": 12, "same": 13
}
id_to_word = {v: k for k, v in vocab.items()}

def decode_sentence(token_ids):
    words = []
    for tid in token_ids.tolist():
        tid = int(tid)
        if tid in id_to_word and tid != 0: # Skip <PAD>
            words.append(id_to_word[tid])
        elif tid >= 20:
            if not words or words[-1] != "[...]": words.append("[...]")
    return " ".join(words)

# 2. THE AMBIGUOUS GENERATOR (Balanced lengths)
def get_logic_batch(batch_size=33, noise_len=200):
    data, targets = [], []
    for i in range(batch_size):
        scenario = i % 3

        # We ensure every scenario results in a total sequence of the same length
        # Start (3 words) + Noise + End (5 words total)
        if scenario == 0:
            # Happy -> Changed (padded to 5 words)
            start, end, target = [3, 1, 4], [7, 8, 9, 10, 0], 0
        elif scenario == 1:
            # Sad -> Changed (padded to 5 words)
            start, end, target = [3, 1, 5], [7, 8, 9, 10, 0], 1
        else:
            # Neutral -> Stayed Same (5 words)
            start, end, target = [3, 1, 4], [7, 8, 11, 12, 13], 1

        noise = [20] * noise_len
        full_seq = start + noise + end
        data.append(torch.tensor(full_seq))
        targets.append(target)

    return torch.stack(data), torch.tensor(targets)

# 3. MODELS
class EmotionStressModel(nn.Module):
    def __init__(self, arch_type, vocab_size=100, embed_dim=16, hidden_dim=16):
        super().__init__()
        self.arch_type = arch_type
        self.embed = nn.Embedding(vocab_size, embed_dim)

        if arch_type == 'RNN':
            self.core = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif arch_type == 'LSTM':
            self.core = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif arch_type == 'Transformer':
            layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=2,
                                               dim_feedforward=32, batch_first=True,
                                               dropout=0.0)
            self.core = nn.TransformerEncoder(layer, num_layers=1)

        self.classifier = nn.Linear(hidden_dim if arch_type != 'Transformer' else embed_dim, 3)

    def forward(self, x):
        x = self.embed(x)
        if self.arch_type in ['RNN', 'LSTM']:
            out, _ = self.core(x)
            return self.classifier(out[:, -1, :])
        else:
            out = self.core(x)
            return self.classifier(out.mean(dim=1))

# 4. EXECUTION
NOISE = 250 # High noise to guarantee RNN failure
print(f"--- STEP 1: LOGIC CHECK (Sentences must be equal length) ---")
sample_in, sample_lab = get_logic_batch(batch_size=3, noise_len=5)
label_names = {0: "Negative", 1: "Positive", 2: "Neutral"}
for i in range(3):
    print(f"Scenario {i+1}: {decode_sentence(sample_in[i])} -> {label_names[sample_lab[i].item()]}")

print("\n--- STEP 2: SHOWDOWN (SGD + AMBIGUOUS ENDINGS) ---")
print(f"{'Architecture':<15} | {'Loss':<10} | {'Test Acc'}")
print("-" * 55)

for arch in ['RNN', 'LSTM', 'Transformer']:
    model = EmotionStressModel(arch)
    # Using SGD ensures no optimizer-based "memory" assistance
    # optimizer = optim.SGD(model.parameters(), lr=0.1)
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for i in range(1500):
        inputs, labels = get_logic_batch(noise_len=NOISE)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    test_in, test_lab = get_logic_batch(batch_size=99, noise_len=NOISE)
    with torch.no_grad():
        acc = (model(test_in).argmax(dim=1) == test_lab).float().mean().item()

    status = "✅ WIN" if acc > 0.80 else "❌ FAIL"
    print(f"{arch:<15} | {loss:.6f} | {acc*100:>6.1f}% {status}")

--- STEP 1: LOGIC CHECK (Sentences must be equal length) ---
Scenario 1: initially i happy [...] but then it changed -> Negative
Scenario 2: initially i sad [...] but then it changed -> Positive
Scenario 3: initially i happy [...] but then stayed the same -> Positive

--- STEP 2: SHOWDOWN (SGD + AMBIGUOUS ENDINGS) ---
Architecture    | Loss       | Test Acc
-------------------------------------------------------
RNN             | 0.462309 |   66.7% ❌ FAIL
LSTM            | 0.462316 |   66.7% ❌ FAIL
Transformer     | 0.000126 |  100.0% ✅ WIN
